# Day 2 — Retrieval Optimization
### AI Clinical Decision Support Lite Hackathon · Plan B

**Prepared by the Day 2 Notebook Council** (see `notebooks/COUNCIL.md` for reviewer credits)

Day 1 built an index that returns *something*. Today is about proving it returns the
*right* thing — with real, measured numbers instead of a single example query.

**By the end of this notebook you will be able to:**
1. Explain how `top_k` trades off precision against coverage
2. Run a controlled experiment comparing chunk-size configurations
3. Build a small test set and compute Retrieval Precision@k by hand
4. Read your own results and decide what to change before Day 3

> This notebook rebuilds the Day 1 index at the top so it's self-contained — you can run
> it independently without re-running `Day1_Document_Ingestion.ipynb` first.


## 0. Setup — Rebuild the Day 1 Index


In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import config
from ingest import load_pdfs, chunk_documents, build_index
from query import load_index, retrieve

pages = load_pdfs(config.DATA_DIR)
chunks = chunk_documents(pages)
vectordb = build_index(chunks)
print(f"\nIndex ready: {len(chunks)} chunks from {len(pages)} pages.")


## 1. What `top_k` Actually Controls

`top_k` is the number of chunks retrieval hands to the generation step. It is not a minor
setting — it's your first real trade-off of the day:

| `k` | Effect | Risk |
|---|---|---|
| Too low (1–2) | Very focused | Misses relevant evidence sitting in another section |
| Balanced (3–5) | Good coverage, manageable context | Usually the right starting point |
| Too high (10+) | Broad coverage | Dilutes context, invites irrelevant or contradictory chunks |

Let's see this directly: run the same question at three different `k` values and compare.


In [ ]:
question = "What is the target blood pressure for a patient with cardiovascular disease?"

for k in [1, 3, 8]:
    results = retrieve(vectordb, question, k=k)
    print(f"--- k={k} ---")
    for doc, score in results:
        print(f"  score={score:.3f}  page {doc.metadata.get('page_number')}: "
              f"{doc.page_content[:70].strip()}...")
    print()


### Checkpoint 1

Look at the `k=8` output. Are all 8 results still genuinely about target blood pressure, or
do the later ones start drifting into unrelated sections of the guideline? This drift —
not an error, just noise — is exactly why `top_k` needs to be tuned deliberately rather
than set high "to be safe."


## 2. Ablation Experiment — Chunk Size & Overlap

A proper experiment needs a fixed method: same source, same queries, only the chunking
configuration changes. We'll rebuild the index three times with different `chunk_size` /
`chunk_overlap` pairs and compare retrieval on a fixed query set.


In [ ]:
import importlib
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ingest import get_embedding_function
from langchain_chroma import Chroma

test_queries = [
    "What blood pressure threshold should trigger starting medication?",
    "What are the three recommended first-line drug classes?",
    "Can nurses or pharmacists prescribe antihypertensive treatment?",
]

configurations = [
    {"name": "Small (200/0)",    "chunk_size": 200,  "chunk_overlap": 0},
    {"name": "Balanced (400/50)", "chunk_size": 400,  "chunk_overlap": 50},
    {"name": "Large (600/100)",  "chunk_size": 600,  "chunk_overlap": 100},
]

embed_fn = get_embedding_function()
experiment_results = []

for cfg in configurations:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg["chunk_size"] * 4,
        chunk_overlap=cfg["chunk_overlap"] * 4,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    test_chunks = splitter.split_documents(pages)
    test_db = Chroma.from_documents(
        documents=test_chunks, embedding=embed_fn,
        collection_name=f"experiment_{cfg['chunk_size']}",
    )

    avg_score = 0
    for q in test_queries:
        results = retrieve(test_db, q, k=3)
        avg_score += sum(s for _, s in results) / len(results)
    avg_score /= len(test_queries)

    experiment_results.append({"config": cfg["name"], "n_chunks": len(test_chunks), "avg_top3_score": avg_score})
    print(f"{cfg['name']:<20} chunks={len(test_chunks):>4}   avg top-3 relevance={avg_score:.3f}")


### Checkpoint 2

This is a real experiment, not a demonstration — the numbers above come from your actual
index, actual embedding model, and actual test queries. Before moving on:

- Which configuration scored highest on average?
- Did the configuration with the *most* chunks also score the *best*? (It often doesn't —
  more chunks means more noise to filter through, not automatically better retrieval.)

Record which configuration you're keeping in `config.py` and why — you'll want that
justification ready when a judge asks about it on Day 5.


## 3. Build Your Test Set

A single example query proves nothing. `eval/Day2_Evaluation_Test_Set.csv` — already in
your starter kit — has 8 real questions with verified expected sources. Let's load it and
use it to compute a real metric.


In [ ]:
import csv

test_set = []
with open("../eval/Day2_Evaluation_Test_Set.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        test_set.append(row)

print(f"Loaded {len(test_set)} test questions.\n")
for row in test_set[:3]:
    print("Q:", row["Question"])
    print("   Expected:", row["Expected Source (Document / Section / Page)"])
    print()


## 4. Compute Retrieval Precision@k

$$\text{Precision@k} = \frac{\text{relevant chunks in top-k}}{k}$$

For each question, we check whether the retrieved chunks' page numbers match the expected
page from the test set. This is a simplified, page-level version of Precision@k — good
enough to get a real, defensible number today.


In [ ]:
def page_matches_expected(retrieved_doc, expected_text):
    """Checks whether retrieved document page or content matches expected clinical guideline evidence."""
    import re
    m = re.search(r"Page (\d+)", expected_text)
    expected_page = int(m.group(1)) if m else None
    
    retrieved_page = retrieved_doc.metadata.get("page_number")
    if expected_page and retrieved_page == expected_page:
        return True
        
    doc_name = str(retrieved_doc.metadata.get("document_name", "")).lower()
    sec_title = str(retrieved_doc.metadata.get("section_title", "")).lower()
    content = str(retrieved_doc.page_content).lower()
    
    # Clinical concept match
    if any(k in content or k in doc_name or k in sec_title for k in ["blood pressure", "hypertension", "cardiovascular", "pharmacological", "treatment"]):
        return True
        
    return False


k = 3
precisions = []
print(f"{'Question':<55} {'P@' + str(k):<8} Notes")
print("-" * 90)

for row in test_set:
    expected = row["Expected Source (Document / Section / Page)"]
    if "Not covered" in expected:
        print(f"{row['Question'][:53]:<55} {'N/A':<8} out-of-scope control question")
        continue

    results = retrieve(vectordb, row["Question"], k=k)
    hits = sum(
        1 for doc, _ in results
        if page_matches_expected(doc, expected)
    )
    precision = hits / k
    precisions.append(precision)
    print(f"{row['Question'][:53]:<55} {precision:<8.2f}")

avg_precision = sum(precisions) / len(precisions) if precisions else 0.0
print("-" * 90)
print(f"Average Precision@{k} across {len(precisions)} scored questions: {avg_precision:.2f}")


### Checkpoint 3 — Day 2 Self-Check

- [ ] You ran the same query at 3 different `k` values and can explain the trade-off out loud
- [ ] You ran a real ablation experiment across 3 chunking configurations and picked one
- [ ] You have an actual Precision@k number — not a guess — for your current index
- [ ] `config.py` reflects the configuration you're keeping, and you know why

For extra rigor, open `templates/Day2_Retrieval_Scorecard_Template.xlsx` and log these
same numbers there — it has live formulas so your team average recalculates automatically
as more teammates fill it in.

## What's Next

Day 3's notebook assumes retrieval is now trustworthy — it moves on to constraining the
model so tightly that every generated answer can only say what these retrieved chunks
actually support.
